# Every AI agent needs a data flywheel
ODSC West 2026 · Nick Ung and Akshay Sharma (Lyft Safety and Customer Care)

One toy support agent (**RideCo Support**: a router plus three subagents, with mock tools and synthetic data). We will **eval it, judge it, break it, and improve it**, one piece of the flywheel at a time:

1. **Harness**: run the agent on templated tasks with a simulated customer and mocked tools
2. **Judge**: an LLM-as-a-judge you can trust (kappa vs expert labels)
3. **Flywheel**: failed traces become regression tasks; verdicts become training data
4. **Close the loop**: fix it, rerun, watch the regression tasks

## 0. Setup

In [ ]:
import os, sys

# Tracing pulls in gRPC, which warns on every subprocess the task runner starts (pink stderr noise).
# These must be set before gRPC is first imported.
os.environ.setdefault("GRPC_ENABLE_FORK_SUPPORT", "0")
os.environ.setdefault("GRPC_VERBOSITY", "NONE")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.path.exists("agent-data-flywheel"):
    !git clone https://github.com/akaysh/agent-data-flywheel.git
    %cd agent-data-flywheel
    !pip -q install -r requirements.txt
sys.path.insert(0, os.getcwd())

In [ ]:
# True  = replay recorded LLM responses. No key, no wifi, same results as on stage.
# False = call a live model (set the endpoint your instructors gave you).
OFFLINE = True  # <-- change to False to call a live model
if "OFFLINE" in os.environ:  # lets scripts override the toggle
    OFFLINE = os.environ["OFFLINE"] == "1"

if not OFFLINE and "LLM_BASE_URL" not in os.environ:
    from getpass import getpass
    os.environ["LLM_BASE_URL"] = input("Base URL (OpenAI-compatible): ")
    os.environ["LLM_API_KEY"] = getpass("API key: ")
    os.environ["AGENT_MODEL"] = input("Agent model (small): ")
    os.environ["FIXED_AGENT_MODEL"] = input("Stronger agent model (the 'swap the model' fix): ")
    os.environ["JUDGE_MODEL"] = input("Judge model (stronger): ")
    os.environ["LLM_TEMPERATURE"] = input("Temperature (Enter for 0, 'none' for reasoning models): ") or "0"
os.environ["OFFLINE"] = "1" if OFFLINE else "0"

**Optional: tracing.** Arize Phoenix runs inside this notebook (no signup). In production, LangSmith or Langfuse play the same role.

In [ ]:
import contextlib, io

PHOENIX_URL = None  # stays None if tracing is skipped
try:
    %pip -q install arize-phoenix openinference-instrumentation-langchain
    import phoenix as px
    from phoenix.otel import register
    from openinference.instrumentation.langchain import LangChainInstrumentor
    with contextlib.redirect_stdout(io.StringIO()):  # Phoenix prints a link that fails (403) on Colab
        session = px.launch_app()
    LangChainInstrumentor().instrument(tracer_provider=register(project_name="rideco"))
    PHOENIX_URL = session.url if session else "http://localhost:6006"  # no session: one is already running
    if IN_COLAB:
        print("Phoenix is running. Run the 'look at the traces' cell below to view it inline.")
    else:
        print("Phoenix UI:", PHOENIX_URL)
except Exception as e:  # tracing is a nice-to-have; the workshop works without it
    print("Skipping tracing:", e)
    print("Local venv without pip? Run: uv pip install arize-phoenix openinference-instrumentation-langchain, then restart the kernel.")

In [ ]:
import asyncio, tempfile
from pathlib import Path
import pandas as pd

from llm.factory import build_chat_model

WORK = Path(tempfile.mkdtemp())
TASKS = Path("tasks")

## 1. Meet the agent
**RideCo Support** answers customers of a fictional rideshare company. A customer writes in, and:

1. A **router** reads the conversation and picks which specialist should handle it. It picks once, on the first message.
2. The **specialist** (one of three subagents) talks to the customer and calls tools. Each one has its own short prompt.
3. **Tools** look things up or take actions: list rides, read charges, refund, escalate to a human.

| Subagent | Handles | Tools it can call |
| --- | --- | --- |
| `charges` | fares, fees, refunds | `list_recent_rides`, `get_ride`, `get_charges`, `issue_refund`, `escalate_to_human` |
| `lost_item` | lost belongings | `list_recent_rides`, `get_ride`, `create_lost_item_ticket`, `contact_driver`, `escalate_to_human` |
| `account` | email and phone changes | `get_account`, `update_account`, `escalate_to_human` |

A subagent only sees its own tools, and all three can hand off to a human. Which tool to use *when* is down to the prompt, and that is what we will test.

### What the prompts say
The shipped agent is **v1**: a prompt someone simplified to sound friendlier and faster. In the process, half the policy was dropped. **v2** is the version with the full policy. Compare them side by side (this is the real text the agent receives).

In [ ]:
from agent.prompts import ROUTER_PROMPTS, SUBAGENT_PROMPTS

print("=== charges agent, v1 (shipped) ===\n" + SUBAGENT_PROMPTS["v1"]["charges"])
print("\n=== charges agent, v2 (fixed) ===\n" + SUBAGENT_PROMPTS["v2"]["charges"])
print("\n=== router, v1 ===\n" + ROUTER_PROMPTS["v1"])

### The tools, and how they are mocked
The tools are served by a small mock backend. It is **deterministic**: each task ships a `fixtures.json` with a few rides, charges and an account, and the tools read from that file. Two things to know:

- **Tools enforce nothing.** `issue_refund` happily refunds $68. Policy (the $25 limit, confirming which ride, never sharing a driver's phone number) is the *agent's* job, and our checks verify whether it kept to it.
- **Every call is logged** to `tool_calls.jsonl`. That log is what the code checks and the judge read afterwards.

In [ ]:
import inspect, json, re
from tools.mock_tools import RideCoTools

tools_table = pd.DataFrame([{
    "tool": name,
    "arguments": re.sub(r"^\(self,? ?", "(", str(inspect.signature(fn))),
    "what it does": inspect.getdoc(fn),
} for name, fn in vars(RideCoTools).items() if inspect.isfunction(fn) and not name.startswith("_")])
pd.set_option("display.max_colwidth", None)
display(tools_table.style.hide(axis="index"))

print("\nExample fixtures for the lost-phone task (the data the tools serve):")
print(Path(TASKS / "lost-phone-basic" / "environment" / "fixtures.json").read_text())

### The customer is simulated too
There are no real customers here. An LLM plays the customer from a **hidden persona**: a goal, facts only the customer knows, and a behavior. The agent never sees the persona, so it has to ask the right questions. Each scenario also defines the **code checks** the verifier runs afterwards (which tools must or must not be called, refund limits).

In [ ]:
import yaml

scenario = next(s for s in yaml.safe_load(open("scenarios/scenarios.yaml"))["scenarios"] if s["id"] == "refund-over-limit")
print("Customer opens with:", scenario["opening_message"])
print("\nHidden persona:\n" + yaml.safe_dump(scenario["persona"], sort_keys=False))
print("Code checks:\n" + yaml.safe_dump(scenario["checks"], sort_keys=False))

### Two sample conversations
Recorded runs of the shipped v1 agent. Both look fine to the customer. Only one actually followed policy.

In [ ]:
from flywheel.traces import load_traces

samples = {t.task_id: t for t in load_traces("data/traces/labeled.jsonl") if t.run_label == "dev-v1"}
for task_id in ("lost-phone-basic", "refund-over-limit"):
    t = samples[task_id]
    print(f"--- {task_id}   verifier reward: {t.reward} ---")
    for turn in t.transcript:
        print(f"{turn['role'].upper():>9}: {turn['content']}")
    print("    TOOLS:", ", ".join(f"{c['tool']}({', '.join(f'{k}={v!r}' for k, v in c['args'].items())})" for c in t.tool_calls))
    print()

The second one is the first planted bug. A duplicate charge of $68 is over the $25 limit, so policy says *escalate to a human, don't refund*. The v1 prompt dropped that rule, the agent refunded, and the customer was told "all set". Now let's run a conversation live.

In [ ]:
from runner.run_tasks import run_task

demo = await run_task(TASKS / "lost-phone-basic", WORK)
for turn in demo.transcript:
    print(f"{turn['role'].upper():>9}: {turn['content']}")
print("\nTool calls:", [c["tool"] for c in demo.tool_calls])
print("Verifier reward:", demo.reward)

## 2. The eval harness
Every scenario is a **Harbor task folder** (`instruction.md`, `environment/`, `tests/`). The same folders run on stage with `harbor run` in Docker and here with a small Docker-free runner. Same tasks, same tests, same scores.

> If you can't replay a conversation the same way twice, you can't measure anything.

In [ ]:
from runner.evaluate import evaluate

before = await evaluate(TASKS, run_label="before-fix", work_root=WORK, prompt_version="v1")

from runner.results import results_rows

def results_table(traces):
    return pd.DataFrame(results_rows(traces, TASKS))

SHOWN = ["task", "pass", "failed checks"]  # the code checks. The judge joins in section 4.
table = results_table(before)
print(f"{table['pass'].sum()}/{len(table)} tasks passed")
table[~table["pass"]][SHOWN]

**Optional: look at the traces.** If the tracing cell above worked, every agent run is now a trace in Phoenix. On Colab the link printed above often shows a 403 error in a new tab, so view it inline here instead.

In [ ]:
if PHOENIX_URL is None:
    print("Tracing is off, skipping.")
elif IN_COLAB:
    from google.colab import output
    output.serve_kernel_port_as_iframe(6006, height=700)
else:
    print("Open Phoenix at", PHOENIX_URL)

## 3. LLM-as-a-judge
Code can check *what the agent did* (refund limits, which tools were called). It can't check *what the agent said*. Look at `escalation-claimed-as-refund`: every code check passes, yet the agent told the customer their refund was done when it only escalated.

A judge is only useful if you can trust it, so we measure it: **Cohen's kappa vs expert labels** (target ≥ 0.7).

In [ ]:
from flywheel.traces import load_traces
from judge.compare import compare_rubrics, load_expert_labels
from judge.run_judge import load_rubric

labeled = load_traces("data/traces/labeled.jsonl")
labels = load_expert_labels("data/expert_labels.csv")
vague_rubric = load_rubric("judge/rubric_vague.yaml")
good_rubric = load_rubric("judge/rubric_good.yaml")
judge_llm = build_chat_model("judge")

print(vague_rubric["prompt"])

In [ ]:
report = await compare_rubrics(judge_llm, judge_llm, vague_rubric, good_rubric, labeled, labels)
print(f"VAGUE rubric (1-10 score)  kappa vs experts: {report['vague']['overall']['kappa']:.2f}")

### Rewrite the rubric
Binary pass/fail per criterion · one criterion per call · precise definition with a pass and a fail example · quote evidence before the verdict · `na` is a real option · never use a judge for something code can check.

In [ ]:
print(good_rubric["criteria"][0]["definition"])
rows = [{"criterion": c, "kappa": r["kappa"], "agreement": r["percent_agreement"], "n": r["n"]}
        for c, r in report["good"].items()]
print(f"GOOD rubric overall kappa vs experts: {report['good']['overall']['kappa']:.2f}")
pd.DataFrame(rows).round(2)

In [ ]:
# The judge-only bug: code checks pass, honesty check fails.
from judge.run_judge import JudgeTrace, judge_good

trace = next(t for t in before if t.task_id == "escalation-claimed-as-refund")
verdicts = await judge_good(judge_llm, good_rubric,
                            JudgeTrace(trace.trace_id, trace.transcript, trace.tool_calls))
print("Verifier reward:", trace.reward)
print("Agent said:", trace.transcript[-1]["content"])
print("honest_about_actions:", verdicts["honest_about_actions"])

## 4. Traces to training data
Every trace is signal. **Failures become regression tasks.** **Judge verdicts become rewards and preference pairs.** We stop at data that is ready to train on; no fine-tuning today.

A **trace** is one finished conversation plus everything we learned about it: the transcript, every tool call, which code checks passed, and the judge's verdict per criterion.

### First, put the judge in the table
So far `pass` meant the code checks only. Now we run the judge on the same traces and add an **`overall pass`** column: a task passes overall only if every code check passed **and** the judge found no failed criterion. The tasks that pass the code checks but fail the judge are the ones code can't see.

In [ ]:
from runner.evaluate import judge_traces

judged = await judge_traces(before, judge_llm, "judge/rubric_good.yaml")
table_judged = results_table(judged)
print(f"code checks only: {table_judged['pass'].sum()}/{len(table_judged)} passed   "
      f"code checks + judge: {table_judged['overall pass'].sum()}/{len(table_judged)} passed")
table_judged[table_judged["pass"] & ~table_judged["overall pass"]][["task", "pass", "judge failed", "overall pass"]]

### Failures become regression tasks
The rule is simple. If **any** code check or judge criterion failed, we copy the scenario that produced the trace into `scenarios/regressions.yaml`:
- Same persona, fixtures and checks, so the task is just as hard as the one that caught the agent.
- New id: `regression-<original id>`. Running it twice adds nothing new.
- `planted_bug` is cleared, and an `origin` records the trace and which checks or criteria failed.
- If the judge failed it, the task's judge criteria shrink to the ones that failed. From then on that task passes overall only if those criteria pass.

`make_tasks.py` then renders it like any other task. The bug you fixed once now has a test that stays in the suite forever.

In [ ]:
from flywheel.regressions import add_regressions

new_ids = add_regressions("scenarios/scenarios.yaml", "scenarios/regressions.yaml", judged)
print(f"{len(new_ids)} regression tasks from failed traces:", new_ids)

In [ ]:
import yaml

regressions = yaml.safe_load(Path("scenarios/regressions.yaml").read_text())["scenarios"]
if regressions:  # a live model that never fails would add none
    example = next((r for r in regressions if r["id"] == "regression-refund-over-limit"), regressions[0])
    print(example["id"], "<-", example["origin"])

In [ ]:
!python scripts/make_tasks.py

## 5. Close the loop
Some of these bugs are *prompt* bugs, one is a *routing* bug, and one is a *model* bug. Let's see which fixes work, and let the tasks tell us.

**Attempt 1: fix the prompts only (v2).** Full policy restored, router lists every agent a conversation needs. Same small model.

In [ ]:
prompt_only = await evaluate(TASKS, run_label="v2-small-model", work_root=WORK, prompt_version="v2")
judged_prompt_only = await judge_traces(prompt_only, judge_llm, "judge/rubric_good.yaml")
table_v2 = results_table(judged_prompt_only)
print(f"code checks only: {table_v2['pass'].sum()}/{len(table_v2)} passed   "
      f"code checks + judge: {table_v2['overall pass'].sum()}/{len(table_v2)} passed")
table_v2[~table_v2["overall pass"]][SHOWN + ["judge failed"]]

The prompt fix moved failures around but didn't remove them. Read the transcripts: the small model **says** "I've sent this to our team" without ever calling `escalate_to_human`. A prompt can't fix that.

**Attempt 2: v2 prompts + a stronger model.** The regression tasks from section 4 are in `tasks/` now.

In [ ]:
after = await evaluate(TASKS, run_label="after-fix", work_root=WORK, prompt_version="v2",
                       agent_llm=build_chat_model("fixed_agent"))
judged_after = await judge_traces(after, judge_llm, "judge/rubric_good.yaml")
compare = results_table(judged).merge(results_table(judged_after), on="task", how="outer",
                                      suffixes=(" (before)", " (after)"))
print(f"overall pass (code checks + judge)   before: {results_table(judged)['overall pass'].sum()}/{len(judged)}"
      f"   after: {results_table(judged_after)['overall pass'].sum()}/{len(judged_after)}")
compare[["task", "overall pass (before)", "overall pass (after)",
         "failed checks (after)", "judge failed (after)"]]

### Verdicts become preference pairs and rewards
**Preference pairs** (for DPO). For each task we take one **good** trace (every code check and every judge criterion passed) and one **bad** trace of the same task. Both start from the same customer message. We cut each conversation at the customer's next message and keep what the agent did in between: the **chosen** reply from the good trace and the **rejected** reply from the bad one. That includes every tool call, not just the final text, because a wrong tool call is what we want to train away. No good trace for a task means no pair, and the output is TRL's conversational format: `{"prompt", "chosen", "rejected"}`.

**Rewards** (for reward modelling or RL). One number per trace: the fraction of its code checks and judge criteria that passed. Criteria marked `na` or `invalid` don't count. A trace that fails one check out of three scores 0.67, not 0, so the signal says how close the agent was.

In [ ]:
from flywheel.preference_pairs import build_preference_pairs, build_reward_records, write_jsonl

pairs = build_preference_pairs(judged + judged_after)
Path("data/export").mkdir(parents=True, exist_ok=True)
write_jsonl(pairs, "data/export/preference_pairs.jsonl")
write_jsonl(build_reward_records(judged + judged_after), "data/export/rewards.jsonl")
print(f"{len(pairs)} preference pairs (TRL DPO format)")

def tools_called(reply):
    return [call["function"]["name"] for m in reply for call in m.get("tool_calls", [])]

if pairs:  # needs a good and a bad trace of the same task
    pair = next((p for p in pairs if p["task_id"] == "refund-over-limit"), pairs[0])
    print("customer said:", pair["prompt"][-1]["content"])
    print("chosen   tools:", tools_called(pair["chosen"]), "->", pair["chosen"][-1]["content"][:90])
    print("rejected tools:", tools_called(pair["rejected"]), "->", pair["rejected"][-1]["content"][:90])

### Same data, other training formats
The traces and verdicts can feed other kinds of training too. Only the shape changes.

| Format | Learns from | Built from our traces |
| --- | --- | --- |
| **SFT** | good examples: "do this" | Keep traces that passed every check and criterion. Each whole conversation is one example. |
| **DPO** | good vs bad of the same task | The preference pairs above. |
| **RLHF** | a reward model trained on preferences, then RL against it | The same pairs train the reward model. Ours come from a judge, not people, so strictly it is RLAIF. |
| **RLVR** | a reward from rules that can't be argued with | The code checks alone. No judge, because a policy will learn to please a judge. |

SFT needs the agent's system prompt in each example, because traces store only the conversation. Pass it as `system_prompt=`.

In [ ]:
from flywheel.training_formats import build_sft_examples, build_verifiable_reward_records

all_traces = judged + judged_after
sft = build_sft_examples(all_traces)
verifiable = build_verifiable_reward_records(all_traces)
write_jsonl(sft, "data/export/sft.jsonl")
write_jsonl(verifiable, "data/export/verifiable_rewards.jsonl")
print(f"{len(sft)} SFT examples out of {len(all_traces)} traces (only the good ones)")
print("RLVR reward spread:", sorted({round(r['reward'], 2) for r in verifiable}))

## Your blueprint
1. **Harness first.** Templated tasks, simulated customer, mocked tools, deterministic verifier.
2. **Code checks for what code can check. Judges for the rest.** Binary criteria, evidence first, kappa ≥ 0.7 against experts.
3. **Every failure becomes a regression task. Every verdict becomes a label.**
4. **Re-run on every prompt, model, or tool change.** The regression tasks tell you whether the fix worked.